# 04. Decompose Patterns — 트렌드 / 계절성 / 잔차 / 이상치

구매 담당자 관점의 시계열 분석. 한 시계열을 다음 4개로 분해합니다.

1. **추세(trend)** — 장기적 방향성 (성장/감소/정체)
2. **계절성(seasonal)** — 12개월 또는 분기 단위 반복 패턴
3. **잔차(residual)** — 위 두 가지로 설명되지 않는 무작위·구조적 변동
4. **이상치(anomalies)** — 잔차의 통계적 분포에서 벗어나는 점

### 왜 이 분해가 중요한가

구매 의사결정에서 "가격이 올랐다"는 표현은 모호합니다. 추세 때문인지, 계절성 때문인지, 일회성 충격인지에 따라 **다른 행동**이 필요합니다.

| 상승의 원인 | 합리적 행동 |
|---|---|
| 추세 (지속적) | 장기 계약·연간 단가 협상 |
| 계절성 (반복) | 비수기에 미리 매입 |
| 잔차 (충격) | 일시 대기 후 평균 회귀 활용 |

### 사용 패키지

- `statsmodels` — STL 분해, ACF/PACF
- `numpy` — z-score 기반 이상치
- `matplotlib` — 시각화 (4단 분해 패널 + 이상치 마킹)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')

## 1. 데이터 로드

`sample_purchases.csv`는 36개월 × 3개 SKU의 매입 단가 시계열입니다. 본인 데이터를 쓸 때는 `INPUT_PATH`만 바꾸세요.

In [ ]:
INPUT_PATH = DATA_DIR / 'sample_purchases.csv'   # 본인 데이터: 'purchases.csv'
df = pd.read_csv(INPUT_PATH, parse_dates=['date']).sort_values('date')
print(df['date'].min().date(), '~', df['date'].max().date())
print('Products:', df['product_id'].unique().tolist())
df.head()

## 2. STL 분해

**STL** (Seasonal-Trend decomposition using LOESS) — 비선형 추세와 변하는 계절성을 잘 다룹니다. 가격 시계열에서 ARIMA 잔차 분석보다 직관적이고 시각화가 쉬워 구매 회의 자료로 적합합니다.

`period=12` (월별 데이터의 연간 계절성). 주별 데이터면 `period=52`.

In [ ]:
TARGET = 'SKU-A001-COTTON'   # 분석할 한 SKU

series = (df[df['product_id'] == TARGET]
          .set_index('date')['unit_cost_krw']
          .asfreq('MS')
          .interpolate())

stl = STL(series, period=12, robust=True).fit()
fig = stl.plot()
fig.set_size_inches(10, 7)
for ax in fig.axes:
    ax.grid(True, alpha=0.3)
fig.suptitle(f'{TARGET} — STL Decomposition', y=1.02)
fig.tight_layout()
fig.savefig(DATA_DIR / f'stl_{TARGET}.png', dpi=120, bbox_inches='tight')
plt.show()

### 해석 포인트

위 4개 패널을 위에서부터 읽습니다.

1. **observed** — 원 시계열 (단가, 원/단위)
2. **trend** — 장기 추세선. 우상향이면 가격 인상 추세 → 장기 계약·헷징 검토.
3. **seasonal** — 매년 반복되는 패턴. 가장 낮은 달(저점)이 **계절적 매입 적기**입니다.
4. **resid** — 잔차. 평균 회귀하는 흐름이라면 비정상적 충격이 곧 정상화될 가능성을 시사.

## 3. 계절 저점 식별

위 `seasonal` 성분에서 **가장 낮은 월**을 찾으면, 이론적으로 그 달이 매입에 가장 유리합니다.

In [ ]:
seasonal_by_month = (
    pd.DataFrame({'seasonal': stl.seasonal.values, 'month': series.index.month})
    .groupby('month')['seasonal'].mean()
    .round(1)
)
best_buy_month = int(seasonal_by_month.idxmin())
worst_buy_month = int(seasonal_by_month.idxmax())
print(seasonal_by_month)
print(f'\n계절 저점(매입 적기): {best_buy_month}월   (평균 영향 {seasonal_by_month.min():.0f}원)')
print(f'계절 고점(매입 비효율): {worst_buy_month}월  (평균 영향 {seasonal_by_month.max():.0f}원)')

## 4. 이상치 탐지 — 잔차의 z-score

잔차 분포가 정규분포에 가깝다고 가정하면, 잔차의 |z| > 2 인 점은 **"통상 패턴으로 설명 안 되는" 가격 충격**입니다.

구매 관점에서:
- |z| > 2 **하방** → 일시적 저가, 매입 기회 후보
- |z| > 2 **상방** → 일시적 고가, 매입 보류 후보 (평균 회귀 기대)

In [ ]:
resid = stl.resid.dropna()
z = (resid - resid.mean()) / resid.std()
anomalies = z[z.abs() > 2.0]
print(f'Anomalies (|z| > 2): {len(anomalies)} points')
anomaly_df = pd.DataFrame({
    'date': anomalies.index,
    'unit_cost_krw': series.loc[anomalies.index].values,
    'residual': resid.loc[anomalies.index].values.round(1),
    'z': anomalies.values.round(2),
    'kind': ['저가 충격' if v < 0 else '고가 충격' for v in anomalies.values],
}).reset_index(drop=True)
anomaly_df

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(series.index, series.values, label='unit cost', color='steelblue')
ax.plot(stl.trend.index, stl.trend.values, label='trend', color='orange', linestyle='--')
if len(anomalies) > 0:
    ax.scatter(anomalies.index, series.loc[anomalies.index],
               color='red', s=80, zorder=5, label='|z|>2 anomalies')
    for d, v, kind in zip(anomalies.index, series.loc[anomalies.index], anomaly_df['kind']):
        ax.annotate(kind, (d, v), textcoords='offset points', xytext=(5, 8), fontsize=9)
ax.set_title(f'{TARGET} — 가격 추세 + 이상치')
ax.set_ylabel('Unit Cost (KRW)')
ax.legend(); ax.grid(True, alpha=0.3)
fig.autofmt_xdate(); fig.tight_layout()
fig.savefig(DATA_DIR / f'anomalies_{TARGET}.png', dpi=120)
plt.show()

## 5. ACF / PACF — 가격이 "기억"하는 정도

**ACF (자기상관)**: 시계열 자기 자신의 과거값과의 상관. 큰 값이 길게 이어지면 **추세형**.

**PACF (편자기상관)**: 직전 시점들의 영향을 제거한 뒤 남은 상관. AR 차수(p) 추정에 사용.

구매 의사결정 관점:
- ACF가 천천히 감쇠 → 가격은 **모멘텀** 성격. 오르면 더 오를 가능성.
- ACF가 빠르게 0 → **노이즈** 성격. 평균 회귀 트레이딩 가능.
- 12개월 주기 spike → 강한 계절성 재확인.

In [ ]:
diff = series.diff().dropna()
max_lag = min(12, max(2, len(diff) // 2 - 1))   # PACF는 표본 50% 미만 lag 요구
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
plot_acf(diff, ax=axes[0], lags=max_lag, title=f'ACF (1차 차분, lags={max_lag})')
plot_pacf(diff, ax=axes[1], lags=max_lag, title=f'PACF (1차 차분, lags={max_lag})')
fig.tight_layout(); plt.show()

## 6. 저장 — Claude Desktop과 공유할 요약

다음 단계 노트북과 `pattern_insights(si)` 프롬프트가 사용할 요약을 markdown으로 저장합니다.

In [ ]:
summary_md = f"""# {TARGET} — Pattern Decomposition Summary

- 분석 기간: {series.index.min().date()} ~ {series.index.max().date()}
- 관측치: {len(series)} 개월

## 추세
- 시작 추세값: {stl.trend.dropna().iloc[0]:.0f} KRW
- 종료 추세값: {stl.trend.dropna().iloc[-1]:.0f} KRW
- 누적 변화율: {(stl.trend.dropna().iloc[-1] / stl.trend.dropna().iloc[0] - 1) * 100:+.1f} %

## 계절성
- 계절 저점(매입 적기): **{best_buy_month}월** (평균 영향 {seasonal_by_month.min():.0f} KRW)
- 계절 고점(매입 비효율): {worst_buy_month}월
- 월별 계절 영향:
{seasonal_by_month.to_string()}

## 이상치 (|z| > 2)
- 발견: {len(anomalies)} 건
- 저가 충격: {(anomaly_df['kind'] == '저가 충격').sum() if len(anomaly_df) else 0} 건
- 고가 충격: {(anomaly_df['kind'] == '고가 충격').sum() if len(anomaly_df) else 0} 건
"""
out = DATA_DIR / f'pattern_summary_{TARGET}.md'
out.write_text(summary_md, encoding='utf-8')
print(f'Saved → {out.resolve()}')
print('\n--- Preview ---\n')
print(summary_md)

## 다음 단계

1. `pattern_summary_*.md`와 `stl_*.png`을 Claude Desktop에 첨부 → `pattern_insights(si)` 프롬프트로 비즈니스 해석.
2. 매크로 영향까지 통합하려면 `05_macro_context.ipynb`.
3. 매입 적기 시점·가격 시그널은 `06_price_signals.ipynb`.